 # Microsoft Fabric Capacity Scaling Up o F128 (ARM) 

This notebook securely authenticates using a service principal, identifies a specific Microsoft Fabric capacity as an Azure resource, scales it to a target SKU using Azure Resource Manager, and waits until the change is complete.

## What it does
1. Configure the Microsoft Fabric capacity that you want to scale
2. Fabric Notebook authenticates as a Service Principal 
3. Notebook confirms the current SKU size 
4. Notebook scales the capacity (if needed) by sending a PATCH request to Azure Resource Manager
5. Re-reads the capacity to confirm the new SKU.

## Prereqs
- Run in a Microsoft Fabric notebook runtime (for `notebookutils`).
- Azure permissions on the capacity resource (Owner/Contributor or a custom role).
- Service Principal configured with permissions & access to Fabric workspace (Admin) 
- Azure Key Vault configured with Service Principal secrets


In [8]:

import json
import time
import requests

StatementMeta(, fa4b7d09-14ea-434b-a86f-8b510632a04a, 10, Finished, Available, Finished)

##### **Assign Microsoft Fabric Environment Information from Azure Portal**
Update the **SUBSCRIPTION_ID**, **RESOURCE_GROUP**, and **CAPACITY_NAME** variables to match the Microsoft Fabric capacity that needs to be scaled up. 

In [9]:
SUBSCRIPTION_ID = "subscription-id"
RESOURCE_GROUP = "resource-group"
CAPACITY_NAME   = "fabric-environment"  # Fabric Capacity Name 
TARGET_SKU      = "F128" # e.g. "F128" or "F64"
API_VERSION     = "2023-11-01" # keep consistent 

StatementMeta(, fa4b7d09-14ea-434b-a86f-8b510632a04a, 11, Finished, Available, Finished)

##### **Assign service principal credential information**
Update the **keyVaultEndpoint** to the Azure Key Vault url and the secret name values if the service principal credentials are stored there.\
These credential information can be hard coded for testing purposes and to get started.

In [10]:
from notebookutils import mssparkutils

keyVaultEndpoint = 'key-vault-name'

tenantId = mssparkutils.credentials.getSecret(keyVaultEndpoint, 'sp-tenant-id')
clientId = mssparkutils.credentials.getSecret(keyVaultEndpoint, 'sp-client-id')
secret = mssparkutils.credentials.getSecret(keyVaultEndpoint, 'sp-client-secret')

StatementMeta(, fa4b7d09-14ea-434b-a86f-8b510632a04a, 12, Finished, Available, Finished)

Py4JJavaError: An error occurred while calling z:mssparkutils.credentials.getSecret.
: com.microsoft.azure.trident.tokenlibrary.util.AkvHttpClientException: Invalid vault uri. Uri should match azure key vault URI like https://<keyVaultName>.vault.azure.net/
	at com.microsoft.azure.trident.tokenlibrary.util.AkvBasedSecretProviderClientImpl.invokeGetTarget(AkvBasedSecretProviderClient.scala:121)
	at com.microsoft.azure.trident.tokenlibrary.util.AkvBasedSecretProviderClientImpl.getAkvSecretWithAccessToken(AkvBasedSecretProviderClient.scala:152)
	at com.microsoft.azure.trident.tokenlibrary.TokenLibrary.getSecretWithToken(TokenLibrary.scala:800)
	at com.microsoft.azure.trident.tokenlibrary.TokenLibrary$.getSecretWithToken(TokenLibrary.scala:1348)
	at mssparkutils.credentials$.getSecret(credentials.scala:166)
	at mssparkutils.credentials.getSecret(credentials.scala)
	at java.base/jdk.internal.reflect.NativeMethodAccessorImpl.invoke0(Native Method)
	at java.base/jdk.internal.reflect.NativeMet

##### **Acquire Tokens and create the API headers**
We need to acquire two tokens:
- PBI audience so that we're able to use the PBI/Fabric APIs.
- Azure Management audience to scale the capacity within Azure.

In [ ]:
from azure.identity import ClientSecretCredential

api_pbi = "https://analysis.windows.net/powerbi/api/.default"
api_arm = "https://management.azure.com/.default"  # <-- use this for ARM

auth = ClientSecretCredential(tenant_id=tenantId, client_id=clientId, client_secret=secret)

header_pbi = {"Authorization": f"Bearer {auth.get_token(api_pbi).token}", "Content-type": "application/json"}
header_arm = {"Authorization": f"Bearer {auth.get_token(api_arm).token}", "Content-type": "application/json"}

StatementMeta(, fa4b7d09-14ea-434b-a86f-8b510632a04a, -1, Cancelled, , Cancelled)

##### **Verify the Current SKU**


In [ ]:
# -----------------------------
# ARM: list capacities
# -----------------------------
list_url = (
    f"https://management.azure.com/subscriptions/{SUBSCRIPTION_ID}"
    f"/providers/Microsoft.Fabric/capacities?api-version={API_VERSION}"
)

r = requests.get(list_url, headers=header_arm)
r.raise_for_status()

capacities = r.json().get("value", [])

# -----------------------------
# Find capacity by name
# -----------------------------
matches = [c for c in capacities if c["name"].lower() == CAPACITY_NAME.lower()]

if not matches:
    raise ValueError(
        f"Capacity '{CAPACITY_NAME}' not found. "
        f"Visible capacities: {[c['name'] for c in capacities]}"
    )

capacity = matches[0]
capacity_id = capacity["id"]

print("Target capacity:", capacity["name"])
print("Resource ID:", capacity_id)

StatementMeta(, fa4b7d09-14ea-434b-a86f-8b510632a04a, -1, Cancelled, , Cancelled)

In [ ]:
# Read current SKU
# -----------------------------
cap_url = f"https://management.azure.com{capacity_id}?api-version={API_VERSION}"

current = requests.get(cap_url, headers=header_arm)
current.raise_for_status()

current_sku = current.json()["sku"]["name"]
print("Current SKU:", current_sku)

StatementMeta(, fa4b7d09-14ea-434b-a86f-8b510632a04a, -1, Cancelled, , Cancelled)

##### **Scale Up to F128**

In [ ]:
import time
# -----------------------------
# Scale if needed
# -----------------------------
if current_sku == TARGET_SKU:
    print(f"No action needed — already on {TARGET_SKU}")
else:
    print(f"Scaling {CAPACITY_NAME} → {TARGET_SKU}")

    payload = {"sku": {"name": TARGET_SKU, "tier": "Fabric"}}
    r = requests.patch(
        cap_url,
        headers=header_arm,
        data=json.dumps(payload)
    )
    r.raise_for_status()

    # Poll until updated
    for _ in range(20):
        time.sleep(10)
        check = requests.get(cap_url, headers=header_arm).json()
        if check["sku"]["name"] == TARGET_SKU:
            print("✅ Scale complete")
            break
    else:
        raise TimeoutError("Scale did not complete in expected time window")

StatementMeta(, fa4b7d09-14ea-434b-a86f-8b510632a04a, -1, Cancelled, , Cancelled)